# 🦅 PREDATOR Analytics v56.1.4-ELITE — 8 DATABASES COLAB DEPLOYMENT

Цей блокнот встановлює **ВЕСЬ СТЕК NVIDIA** на Google Colab:
1. **PostgreSQL 16** (Core DB)
2. **Redis 7** (Cache / PubSub)
3. **Neo4j 5** (Графова БД)
4. **OpenSearch 2.12** (Текстовий пошук)
5. **Qdrant 1.8** (Векторна БД)
6. **MinIO** (Об'єктне сховище S3)
7. **Kafka + Zookeeper** (Черга повідомлень)
8. **Ollama + LiteLLM** (AI Gateway)

+ FastAPI Backend + Nginx + ZROK тунель.

> ⚠️ Зверніть увагу: запуск усіх цих інструментів у Colab займе близько 5–7 хвилин і споживе ~6-8 ГБ RAM.

In [ ]:
%%bash
# ==========================================================
# СИСТЕМНІ ЗАЛЕЖНОСТІ ТА JAVA 17 (для Neo4j, OpenSearch, Kafka)
# ==========================================================
echo "[1/6] Встановлення базових пакетів..."
apt-get update -qq
apt-get install -y -qq postgresql-16 redis-server openjdk-17-jdk curl wget jq sudo 2>/dev/null

In [ ]:
%%bash
# ==========================================================
# 8 БАЗ ДАНИХ ТА СЕРВІСІВ (The 8 Databases)
# ==========================================================
echo "[2/6] Запуск 8 баз даних..."

# 1. PostgreSQL (5432)
service postgresql start 2>/dev/null; sleep 3
su -c "psql -c \"CREATE USER predator WITH PASSWORD 'predator2026';\" 2>/dev/null" postgres
su -c "psql -c \"CREATE DATABASE predator_db OWNER predator;\" 2>/dev/null" postgres
echo "✅ PostgreSQL запущено!"

# 2. Redis (6379)
service redis-server start 2>/dev/null
echo "✅ Redis запущено!"

# 3. Neo4j (7474, 7687)
echo "Завантаження Neo4j..."
wget -qO neo4j.tar.gz "https://neo4j.com/artifact.php?name=neo4j-community-5.23.0-unix.tar.gz"
tar -xzf neo4j.tar.gz
mv neo4j-community-5.23.0 /opt/neo4j
# Вимикаємо вимогу аутентифікації для простоти дев-середовища
sed -i 's/#dbms.security.auth_enabled=false/dbms.security.auth_enabled=false/' /opt/neo4j/conf/neo4j.conf
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
/opt/neo4j/bin/neo4j start > /var/log/neo4j.log 2>&1
echo "✅ Neo4j запущено!"

# 4. OpenSearch (9200)
echo "Завантаження OpenSearch..."
wget -qO opensearch.tar.gz "https://artifacts.opensearch.org/releases/bundle/opensearch/2.12.0/opensearch-2.12.0-linux-x64.tar.gz"
tar -xzf opensearch.tar.gz -C /opt/
useradd -m opensearch_user || true
chown -R opensearch_user:opensearch_user /opt/opensearch-2.12.0
echo "discovery.type: single-node" >> /opt/opensearch-2.12.0/config/opensearch.yml
echo "plugins.security.disabled: true" >> /opt/opensearch-2.12.0/config/opensearch.yml
sudo -u opensearch_user -E nohup /opt/opensearch-2.12.0/bin/opensearch -d > /var/log/opensearch.log 2>&1 &
echo "✅ OpenSearch запущено!"

# 5. Qdrant (6333, 6334)
echo "Завантаження Qdrant..."
wget -qO qdrant.tar.gz "https://github.com/qdrant/qdrant/releases/download/v1.8.0/qdrant-x86_64-unknown-linux-gnu.tar.gz"
tar -xzf qdrant.tar.gz -C /usr/local/bin/
nohup qdrant > /var/log/qdrant.log 2>&1 &
echo "✅ Qdrant запущено!"

# 6. MinIO (9000)
echo "Завантаження MinIO..."
wget -qO /usr/local/bin/minio "https://dl.min.io/server/minio/release/linux-amd64/minio"
chmod +x /usr/local/bin/minio
mkdir -p /data/minio
export MINIO_ROOT_USER=predator
export MINIO_ROOT_PASSWORD=predator2026
nohup minio server /data/minio --address ":9000" --console-address ":9001" > /var/log/minio.log 2>&1 &
echo "✅ MinIO запущено!"

# 7. Kafka & Zookeeper (9092, 2181)
echo "Завантаження Kafka..."
wget -qO kafka.tgz "https://archive.apache.org/dist/kafka/3.7.0/kafka_2.13-3.7.0.tgz"
tar -xzf kafka.tgz -C /opt/
nohup /opt/kafka_2.13-3.7.0/bin/zookeeper-server-start.sh /opt/kafka_2.13-3.7.0/config/zookeeper.properties > /var/log/zookeeper.log 2>&1 &
sleep 5
nohup /opt/kafka_2.13-3.7.0/bin/kafka-server-start.sh /opt/kafka_2.13-3.7.0/config/server.properties > /var/log/kafka.log 2>&1 &
echo "✅ Kafka + Zookeeper запущено!"

# 8. Ollama + LiteLLM (11434, 4000)
echo "Завантаження Ollama..."
curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
nohup ollama serve > /var/log/ollama.log 2>&1 &
pip install -q litellm[proxy]
echo "✅ Ollama (Local AI) запущено!"


In [ ]:
%%bash
# ==========================================================
# СТВОРЕННЯ МАКЕТУ BACKEND ТА UI 
# ==========================================================
echo "[3/6] Налаштування FastAPI..."
pip install -q fastapi 'uvicorn[standard]' psycopg2-binary redis httpx neo4j

mkdir -p /opt/predator-api
cat > /opt/predator-api/main.py << 'PYEOF'
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from datetime import datetime

app = FastAPI(title="PREDATOR Analytics API 8-DB Edition")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/")
def root(): return {"status": "PREDATOR v56.1.4-ELITE", "timestamp": datetime.now().isoformat()}

@app.get("/api/v1/azr/status")
def status(): 
    return {
        "status": "operational",
        "mode": "colab_8db",
        "dbs_active": ["postgres", "redis", "neo4j", "opensearch", "qdrant", "kafka", "minio", "ollama"]
    }

@app.get("/health")
def health(): return {"status": "ok"}

@app.post("/api/v1/auth/login")
def login(c: dict): 
    return {"access_token": "token", "token_type": "bearer", "user": {"username": "admin", "role": "admin"}}

@app.get("/api/v1/dashboard/overview")
def overview(): return {"total_companies": 8, "active_declarations": 1024, "total_value_usd": 50000000, "risk_alerts": 4, "mode": "colab"}
PYEOF

pkill -f uvicorn || true
nohup python3 -m uvicorn main:app --host 0.0.0.0 --port 8000 --workers 2 > /var/log/predator-api.log 2>&1 &

echo "[4/6] Налаштування Nginx..."
apt-get install -y -qq nginx > /dev/null
mkdir -p /opt/predator-ui
echo "API OK. 8 DBs Running." > /opt/predator-ui/index.html
cat > /etc/nginx/sites-available/predator << 'NGINX'
server {
    listen 3030;
    root /opt/predator-ui;
    index index.html;
    location /api/ { proxy_pass http://localhost:8000; proxy_set_header Host $host; add_header 'Access-Control-Allow-Origin' '*'; }
    location /health { proxy_pass http://localhost:8000/health; }
    location / { try_files $uri $uri/ /index.html; }
}
NGINX
ln -sf /etc/nginx/sites-available/predator /etc/nginx/sites-enabled/predator
rm -f /etc/nginx/sites-enabled/default
service nginx restart
echo "✅ API та Nginx готові!"


In [ ]:
%%bash
# ==========================================================
# ZROK ТУНЕЛЬ (predator зарезервований)
# ==========================================================
echo "[5/6] Запуск zrok..."
wget -q https://github.com/openziti/zrok/releases/download/v0.4.45/zrok_0.4.45_linux_amd64.tar.gz -O /tmp/zrok.tar.gz
tar -xzf /tmp/zrok.tar.gz --wildcards '*/zrok' -O > /usr/local/bin/zrok
chmod +x /usr/local/bin/zrok
zrok disable 2>/dev/null || true
zrok enable 1eeje4um7yvA > /dev/null 2>&1

pkill -f zrok || true
rm -f /tmp/zrok.log
nohup zrok share reserved predator --headless > /tmp/zrok.log 2>&1 &
sleep 10

echo ""
echo "╔══════════════════════════════════════════════════════════╗"
echo "║ 🦅 PREDATOR Analytics — 8 DBs COLAB CLUSTER READY!       ║"
echo "╠══════════════════════════════════════════════════════════╣"
echo "║ 🌐 Frontend / API URL:  https://predator.share.zrok.io   ║"
echo "║                                                          ║"
echo "║ ✅ Бази даних що зараз працюють у фоні Colab:            ║"
echo "║  - PostgreSQL 16 (5432)                                  ║"
echo "║  - Redis 7 (6379)                                        ║"
echo "║  - Neo4j 5 Графи (7687)                                  ║"
echo "║  - OpenSearch 2.12 (9200)                                ║"
echo "║  - Qdrant 1.8 Векори (6333)                              ║"
echo "║  - MinIO Об'єкти (9000)                                  ║"
echo "║  - Kafka + Zookeeper (9092)                              ║"
echo "║  - Ollama AI (11434)                                     ║"
echo "╚══════════════════════════════════════════════════════════╝"


In [ ]:
# ==========================================================
# KEEP ALIVE COLAB
# ==========================================================
import time, datetime
print("✅ Кластер утримується активним...")
while True:
    print(f'\r⏱️ {datetime.datetime.now().strftime("%H:%M:%S")} | 8 DB CLUSTER ONLINE', end='', flush=True)
    time.sleep(60)
